# Tissue-Level Expression: GTEx


## What does GTEx add?

The source paper gives us 54 variants in 25 genes. Our next question is whether
those genes are expressed in heart tissue.

GTEx stands for **Genotype-Tissue Expression**. This NIH Common Fund resource
measures gene expression across human tissues. This lesson uses median
transcripts per million, or TPM, from heart atrial appendage and left
ventricle.

GTEx uses bulk tissue, so each value combines signals from many cell types. It
adds tissue context to a gene. It does not test the effect of a specific
variant or change the class reported by the source paper.

## How the GTEx API Works

The live request first matches the paper's 25 gene symbols to GENCODE v39 IDs.
A second request asks for GTEx v10 median expression in two heart tissues. The
helper returns the same columns used by the rest of the lesson.

First, load the published variants and the shared API helper.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd() if Path("api_helpers.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from api_helpers import fetch_gtex_context

DATA_DIR = REPO_ROOT / "data"
variants = pd.read_csv(DATA_DIR / "variants.csv")


Next, query GTEx for the 25 unique genes. The commented line loads the saved
response if the live service is unavailable.


In [ ]:
gene_symbols = sorted(variants["gene_symbol"].unique())
gtex = fetch_gtex_context(gene_symbols)

# Backup: use the frozen 2026-08-11 response instead of the live API.
# gtex = pd.read_csv(DATA_DIR / "gtex_expression.csv")

gtex.head()


### Live data and backup
The default code queries GTEx. Live values can change. If the request fails,
comment out the live line and uncomment the saved-data line.

## Compare Heart Tissues

Run the same tissue comparison used in the GTEx notebook.

First, reshape the GTEx values and add the number of published variant rows
connected to each gene.


In [ ]:
variant_counts = variants.groupby("gene_symbol").size().rename("variant_rows")
heart_expression = (
    gtex.pivot(
        index="gene_symbol",
        columns="tissue_name",
        values="median_tpm",
    )
    .join(variant_counts)
    .sort_values("Heart - Left Ventricle", ascending=False)
)
heart_expression.head(10)


Next, plot the ten genes with the highest median left-ventricle expression.
The full 25-gene table remains available above.


In [ ]:
top_heart_genes = heart_expression.nlargest(10, "Heart - Left Ventricle")
axis = top_heart_genes.loc[
    :, ["Heart - Atrial Appendage", "Heart - Left Ventricle"]
].plot.bar(
    color=["#3d64b3", "#764c82"],
    figsize=(10, 5),
)
axis.set_ylabel("Median expression (TPM)")
axis.set_xlabel("Gene from the source paper")
axis.set_title("GTEx v10 heart-tissue expression")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
